<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L08-pipeline-drift-monitor/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L08-pipeline-drift-monitor/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/document-intelligence/lessons/P02-L08-pipeline-drift-monitor/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/document-intelligence/lessons/P02-L08-pipeline-drift-monitor/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P02-L08 · Drift monitoring for a live document pipeline

**You will build:** the monitor for module 1's extraction pipeline once it is live — a
population stability index and a KL divergence over field-value and confidence distributions,
a daily quality proxy from a random audit, a CUSUM chart on that proxy, and a paging threshold
chosen against an explicit false-page budget instead of a round number.

**Time:** ~70 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download, no model
API · **Prerequisites:** T00-L01 (the 8 GB track), P02-L01 (the evaluation harness).

You will replay one synthetic year of traffic through a monitor that did not exist while it
happened. Three things changed that year: a large supplier switched to a new invoice template,
the document mix shifted around the half-year close, and a model update shipped a real
regression. Only one of them should wake anybody up. By the end you will be able to:

1. Implement a population stability index and a KL divergence over binned field-value and
   confidence distributions, on edges cut once on a baseline and floored on both sides.
2. Implement the audited defect rate as a daily quality proxy, and a one-sided CUSUM on it.
3. Measure the false-page rate of a CUSUM threshold on simulated no-change years, and choose
   the most sensitive threshold inside an explicit budget instead of a round number.
4. Replay the year, catch the regression without paging for the other two changes, and tell
   the three apart by which distribution moved and whether quality moved.
5. Explain why a drift statistic, or a label-free confidence proxy, pages for the harmless
   changes and sleeps through the harmful one.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import sys
import time
import traceback
from typing import Callable, Mapping, NamedTuple, Sequence

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

SEED = 20260923
N_DAYS = 364              # one replay year: four quarters of QUARTER_DAYS, fifty-two weeks
QUARTER_DAYS = 91
DOCS_PER_DAY = 200        # documents the pipeline extracts every day
AUDIT_DOCS = 25           # of those, documents a reviewer checks end to end, chosen at random
BASELINE_DAYS = 91        # the first quarter: the monitor is calibrated here and nowhere else
MONITOR_START = BASELINE_DAYS
WINDOW_DAYS = 28          # the weekly drift report looks back this many days
N_BINS = 10
PSI_FLOOR = 1e-6          # the floor on a bin's share, the same one the model-risk programme uses

# Policy inputs. Nothing here is a fact about the world; each is a decision someone owns, and the
# lesson measures what each one costs rather than asserting that it is right.
PSI_RULE_OF_THUMB = 0.25  # the conventional "significant change" band for PSI (claims.yaml)
FALSE_PAGE_BUDGET = 1.0   # false pages per quarter the on-call rota has agreed to absorb
MIN_SHIFT = 0.02          # the smallest rise in the audited defect rate that must page someone
NULL_YEARS = 200          # simulated no-change years behind every false-page rate
H_GRID = tuple(float(h) for h in np.round(np.arange(1.0, 8.01, 0.25), 2))
ROUND_THRESHOLDS = (4.0, 5.0)   # the rule-of-thumb h a handbook suggests (claims.yaml)
DRIFT_DRAWS = 100         # simulated no-change windows behind each drift level
DRIFT_FLAG_RATE = 0.01    # share of no-change windows allowed to flag "moved"

# Module 1's vocabulary, carried over unchanged: the six fields, and the five things an audit
# can say about one (document, field) cell.
SCHEMA: dict[str, str] = {
    "invoice_id": "id",
    "invoice_date": "date",
    "total_amount": "money",
    "currency": "id",
    "counterparty": "text",
    "payment_terms_days": "integer",
}
FIELDS = tuple(SCHEMA)
ERROR_LABELS = ("correct", "miss", "spurious", "wrong_value", "true_negative")
DEFECT_LABELS = ("miss", "spurious", "wrong_value")   # the three that are the extractor's fault
NOT_AUDITED = -1          # verdict code for a cell nobody reviewed

CURRENCIES = ("EUR", "USD", "GBP")
TERMS = ("", "14", "30", "45", "60")                   # "" = no payment terms on the page
# The signals the drift report watches. Field VALUES the pipeline emitted, and the CONFIDENCE it
# attached to each field. Two value signals are categorical: the number is a code, not a size.
VALUE_SIGNALS = ("total_amount", "currency", "payment_terms_days")
CONFIDENCE_SIGNALS = tuple(f"confidence:{f}" for f in FIELDS)
SIGNALS = VALUE_SIGNALS + CONFIDENCE_SIGNALS
CATEGORIES: dict[str, int] = {"currency": len(CURRENCIES), "payment_terms_days": len(TERMS)}
TRIAGE_LABELS = ("regression", "template change", "mix shift",
                 "template change and mix shift", "no change")


class StabilityResult(NamedTuple):
    """PSI and the per-bin arithmetic that produced it, so a finding can be traced to a bin."""
    psi: float                   # the total
    contributions: np.ndarray    # per-bin (a - e) * ln(a / e), same length as the bins
    expected_pct: np.ndarray     # baseline share per bin, summing to 1.0
    actual_pct: np.ndarray       # current share per bin, summing to 1.0


class Drift(NamedTuple):
    """One signal's movement against the baseline, in the weekly drift report."""
    psi: float                   # symmetric: how far apart the two distributions are
    kl: float                    # KL(current || baseline), nats: how surprising today is


class CusumResult(NamedTuple):
    statistic: np.ndarray        # the CUSUM value computed on each day, before any reset
    pages: tuple                 # indices of the days on which it exceeded h


class Threshold(NamedTuple):
    """A paging threshold that states what it costs."""
    h: float
    false_pages_per_quarter: float   # MEASURED on simulated no-change years
    budget: float                    # the most the on-call rota agreed to


class Triage(NamedTuple):
    label: str                   # one of TRIAGE_LABELS
    page: bool                   # wake somebody up?


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("population_stability_index",),
    "exercise 2": ("kl_divergence",),
    "exercise 3": ("fit_edges", "drift_scores"),
    "exercise 4": ("daily_defect_rate",),
    "exercise 5": ("cusum",),
    "exercise 6": ("false_pages_per_quarter", "choose_threshold"),
    "exercise 7": ("triage",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 1"] -> "exercise 1 (population_stability_index)"; several -> "exercises 2, 3
    and 5"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)

## 1. The year you are about to replay

Module 1 built the harness for a remittance-advice extractor. That extractor has now been live
for a year, reading `DOCS_PER_DAY` documents a day. Every day a reviewer also checks
`AUDIT_DOCS` of them end to end, chosen **at random**, and writes one of module 1's
`ERROR_LABELS` against every cell. Nobody reads the other documents twice.

`simulate_year()` generates that year from a fixed seed, and `EVENTS` lists what happened in
it. Read the generator: it says exactly what each event does to the data, which is what makes
every later verdict in this lesson checkable. Then run it and read the table it prints.

In [ ]:
class Event(NamedTuple):
    name: str
    kind: str       # "mix" | "regression" | "template"
    start: int      # first day it is live
    end: int        # first day it is over


EVENTS = (
    Event("half-year-end mix shift", "mix", 140, 182),
    Event("model regression", "regression", 210, 266),   # rolled back after a customer complaint
    Event("vendor template change", "template", 294, N_DAYS),
)

SUPPLIERS = ("Nordwind Logistik GmbH", "Vantor Marine B.V.", "Helix Pharma Limited",
             "Caldera Energy PLC", "Kestrel Freight Inc", "Meridian Custody AG")
TEMPLATE_VENDOR = 0                                      # Nordwind is the one that re-templates
_SUPPLIER_SHARES = (0.30, 0.18, 0.16, 0.14, 0.12, 0.10)
_CURRENCY_SHARES = {"usual": (0.50, 0.30, 0.20), "mix": (0.30, 0.52, 0.18)}
_TERMS_SHARES = (0.28, 0.12, 0.34, 0.12, 0.14)
_LOG10_AMOUNT = (3.2, 0.55)                              # mean and sd of log10(amount)
_MIX_AMOUNT_SHIFT = 0.45                                 # the half-year close brings bigger bills
_FIELD_ERRORS = {"invoice_id": (0.05, 0.04), "invoice_date": (0.05, 0.05),   # (miss, wrong)
                 "total_amount": (0.04, 0.05), "currency": (0.02, 0.03),
                 "counterparty": (0.04, 0.09), "payment_terms_days": (0.08, 0.05)}
_P_SPURIOUS = 0.20                                       # invents terms that were never printed
_REGRESSION_EXTRA_WRONG = {"total_amount": 0.10, "invoice_date": 0.08}
_TEMPLATE_CONFIDENCE_DROP = 0.30


class PipelineLog(NamedTuple):
    """One year of traffic, one row per document, columns as numpy arrays.

    Module 1 scored one dict per document. A year is DOCS_PER_DAY x N_DAYS documents, so this
    log is columnar instead, and keeps module 1's vocabulary: SCHEMA's fields, ERROR_LABELS.
    """
    day: np.ndarray        # (n_docs,) day index, sorted
    supplier: np.ndarray   # (n_docs,) index into SUPPLIERS
    signals: dict          # signal name -> (n_docs,) what the pipeline EMITTED (see SIGNALS)
    verdict: np.ndarray    # (n_docs, n_fields) int8 code into ERROR_LABELS, NOT_AUDITED if unread
    truth: np.ndarray      # (n_docs, n_fields) the verdict every cell WOULD get. Production never
                           # has this column; the lesson keeps it only to grade the monitor.


def simulate_year(seed: int = SEED, events: Sequence[Event] = EVENTS) -> PipelineLog:
    """A deterministic year of pipeline traffic with the given events switched on.

    What each event does, and nothing else:
      * mix        — more USD invoices and larger amounts. The extractor is exactly as good.
      * regression — extra wrong values on `total_amount` and `invoice_date`, written with the
                     confidence of a CORRECT answer: the new model is confidently wrong.
      * template   — every field the template vendor's documents carry is read with lower
                     confidence. The extractor is exactly as good.

    Every random number is drawn up front, whatever `events` says, so two calls with the same
    seed differ ONLY in what the events changed. That is what lets section 9 ask of any page,
    "would it have happened without this event?", and get an exact answer.
    """
    rng = np.random.default_rng(seed)
    n, n_f = N_DAYS * DOCS_PER_DAY, len(FIELDS)
    day = np.repeat(np.arange(N_DAYS), DOCS_PER_DAY)
    u_sup, u_cur, u_terms = rng.random(n), rng.random(n), rng.random(n)
    z_amount = rng.standard_normal(n)
    u_verdict, u_conf, u_wrong = rng.random((n, n_f)), rng.random((n, n_f)), rng.random((n, n_f))
    audit_rank = rng.random((N_DAYS, DOCS_PER_DAY)).argsort(axis=1).argsort(axis=1)
    audited = (audit_rank < AUDIT_DOCS).ravel()

    def live(kind: str) -> np.ndarray:
        on = np.zeros(n, dtype=bool)
        for e in events:
            if e.kind == kind:
                on |= (day >= e.start) & (day < e.end)
        return on

    mix, regression, template = live("mix"), live("regression"), live("template")
    pick = lambda shares, u: np.searchsorted(np.cumsum(shares)[:-1], u, side="right")  # noqa: E731
    supplier = pick(_SUPPLIER_SHARES, u_sup)
    currency = np.where(mix, pick(_CURRENCY_SHARES["mix"], u_cur),
                        pick(_CURRENCY_SHARES["usual"], u_cur))
    log_amount = _LOG10_AMOUNT[0] + _LOG10_AMOUNT[1] * z_amount + _MIX_AMOUNT_SHIFT * mix
    terms = pick(_TERMS_SHARES, u_terms)                # 0 = no terms printed

    code = {label: i for i, label in enumerate(ERROR_LABELS)}
    truth = np.empty((n, n_f), dtype=np.int8)
    conf = np.empty((n, n_f))
    for j, field in enumerate(FIELDS):
        p_miss, p_wrong = _FIELD_ERRORS[field]
        extra = _REGRESSION_EXTRA_WRONG.get(field, 0.0) * regression
        u = u_verdict[:, j]
        label = np.where(u < p_miss, code["miss"],
                         np.where(u < p_miss + p_wrong + extra, code["wrong_value"], code["correct"]))
        if field == "payment_terms_days":
            label = np.where(terms == 0, np.where(u < _P_SPURIOUS, code["spurious"],
                                                  code["true_negative"]), label)
        c = np.select([label == code["correct"], label == code["miss"],
                       label == code["wrong_value"], label == code["spurious"]],
                      [0.62 + 0.37 * u_conf[:, j], 0.0 * u_conf[:, j],
                       0.34 + 0.52 * u_conf[:, j], 0.30 + 0.42 * u_conf[:, j]],
                      default=0.80 + 0.18 * u_conf[:, j])
        confidently_wrong = (label == code["wrong_value"]) & (u >= p_miss + p_wrong)
        c = np.where(confidently_wrong, 0.62 + 0.37 * u_conf[:, j], c)
        c = np.where(template & (supplier == TEMPLATE_VENDOR),
                     np.clip(c - _TEMPLATE_CONFIDENCE_DROP, 0.0, 1.0), c)
        truth[:, j], conf[:, j] = label, np.round(c, 3)

    # What the pipeline EMITTED. A wrong amount is off by a digit or more, a missed one is blank.
    ja, jc, jt = (FIELDS.index(f) for f in ("total_amount", "currency", "payment_terms_days"))
    off = np.where(u_wrong[:, ja] < 0.5, -1.0, 1.0) * (0.05 + 0.95 * u_wrong[:, ja])
    amount = np.where(truth[:, ja] == code["wrong_value"], log_amount + off, log_amount)
    amount = np.where(truth[:, ja] == code["miss"], np.nan, amount)
    cur = np.where(truth[:, jc] == code["wrong_value"],
                   (currency + 1 + (u_wrong[:, jc] < 0.5)) % len(CURRENCIES), currency)
    cur = np.where(truth[:, jc] == code["miss"], -1, cur)
    t = truth[:, jt]
    wrong_terms = 1 + (terms + (u_wrong[:, jt] * 3).astype(int)) % 4      # a different term
    invented = 1 + (u_wrong[:, jt] * 4).astype(int)
    emitted_terms = np.select([t == code["correct"], t == code["wrong_value"],
                               t == code["spurious"]], [terms, wrong_terms, invented], default=0)
    signals = {"total_amount": amount, "currency": cur, "payment_terms_days": emitted_terms}
    for j, field in enumerate(FIELDS):
        signals[f"confidence:{field}"] = conf[:, j]
    verdict = np.where(audited[:, None], truth, NOT_AUDITED).astype(np.int8)
    return PipelineLog(day, supplier, signals, verdict, truth)


def _signals_at(log: PipelineLog, rows: np.ndarray) -> dict[str, np.ndarray]:
    """The emitted signals of the given document rows, blanks left out: a missed amount or
    currency has no value to put in a histogram."""
    out = {}
    for name in SIGNALS:
        x = log.signals[name][rows]
        if name == "total_amount":
            x = x[~np.isnan(x)]
        elif name == "currency":
            x = x[x >= 0]
        out[name] = x
    return out


def window_signals(log: PipelineLog, first_day: int, last_day: int) -> dict[str, np.ndarray]:
    """Every signal for the documents of days first_day..last_day inclusive."""
    return _signals_at(log, np.arange(first_day * DOCS_PER_DAY, (last_day + 1) * DOCS_PER_DAY))


LOG = simulate_year()
# The same year with one event switched off at a time: identical random numbers, so any
# difference between LOG and one of these is caused by that event and by nothing else.
COUNTERFACTUALS = {e.kind: simulate_year(events=tuple(x for x in EVENTS if x is not e))
                   for e in EVENTS}
print(f"{len(LOG.day)} documents over {N_DAYS} days; {int((LOG.verdict[:, 0] >= 0).sum())} "
      f"of them audited, {AUDIT_DOCS} a day.\n")
for e in EVENTS:
    print(f"  day {e.start:3d} to {e.end - 1:3d}  {e.name}")
print(f"\n{'days':>9s}{'USD share':>11s}{'median amount':>15s}{'mean confidence':>17s}")
_conf_all = np.column_stack([LOG.signals[s] for s in CONFIDENCE_SIGNALS])
for _b in range(0, N_DAYS, WINDOW_DAYS):
    _w = window_signals(LOG, _b, _b + WINDOW_DAYS - 1)
    _rows = slice(_b * DOCS_PER_DAY, (_b + WINDOW_DAYS) * DOCS_PER_DAY)
    print(f"{_b:4d}-{_b + WINDOW_DAYS - 1:<4d}{np.mean(_w['currency'] == 1):11.3f}"
          f"{10 ** np.median(_w['total_amount']):15,.0f}{_conf_all[_rows].mean():17.3f}")
print("\nThe mix shift is in the currency and amount columns and the template change is in the")
print("confidence column. The regression is in none of them: the new model is confidently wrong.")

## 2. Exercise 1 — `population_stability_index`

A drift monitor answers "is today's traffic still the traffic I calibrated on?" one signal at
a time. PSI compares the *shape* of a distribution against a baseline, bin by bin, over
shares of each sample's own total:

`PSI = Σ (a_i − e_i) · ln(a_i / e_i)`

Use exactly the conventions of the model-risk programme's validation suite, so a PSI from
either course means the same thing: bins `[edges[i], edges[i+1])`, the last closed on the
right and anything beyond the outer edges clipped into the end bins; shares, not counts; every
share floored at `PSI_FLOOR` on BOTH sides before the logarithm.

<details><summary>💡 Hint 1 — what to think about</summary>

PSI compares shapes, not sizes: a baseline quarter holds far more documents than a four-week
window, so each side must become shares of its OWN total. Then look hard at the logarithm. A
bin can empty out on either side — a band that vanishes, or one that appears from nothing —
and either one sends the ratio to zero or to infinity. Those are the loudest findings PSI has.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Reject edges with fewer than two values, or with any step that is not strictly upward. Find
each value's bin by searching the edges from the right and stepping back one, then clip the
index into the valid range. Count per bin on each side and divide by that side's own length.
Raise every share on BOTH sides to at least the floor, form each bin's term, and return the
terms, their sum and both unfloored share vectors.

</details>

In [ ]:
def population_stability_index(expected: np.ndarray, actual: np.ndarray,
                               edges: np.ndarray, floor: float = PSI_FLOOR) -> StabilityResult:
    """Population stability index of `actual` against baseline `expected`, on given `edges`.

    `edges` has `len(edges) - 1` bins; bin *i* is `[edges[i], edges[i+1])`, and the last bin
    is closed on the right. Values outside the edges fall into the nearest end bin, which is
    why `quantile_edges` opens the outer edges to infinity.

    Requirements, each graded:
      * shares, not counts: each side is divided by its OWN total, so samples of different
        sizes are comparable.
      * every share is floored at `floor` before the logarithm. A bin that empties out is the
        single most important thing PSI can tell you, and `log(0)` throws it away.
      * `contributions` is the per-bin term and sums to `psi`; each term is non-negative.
        `expected_pct` and `actual_pct` are the shares BEFORE flooring, each summing to 1.
      * `ValueError` if `edges` is not strictly increasing, or holds fewer than two values.

    Example — identical distributions shift nothing:
        >>> e = np.array([0.1, 0.2, 0.8, 0.9])
        >>> round(population_stability_index(e, e, np.array([0.0, 0.5, 1.0])).psi, 12)
        0.0
    Returns:
        A StabilityResult(psi, contributions, expected_pct, actual_pct).
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_psi() -> None:
    same = np.array([0.1, 0.2, 0.8, 0.9])
    flat = population_stability_index(same, same, np.array([0.0, 0.5, 1.0]))
    assert isinstance(flat, StabilityResult), "return a StabilityResult, not a bare float"
    assert abs(flat.psi) < 1e-12, (
        f"two identical samples must score 0.0, got {flat.psi:.6f} — are you comparing counts "
        "instead of shares?")
    sized = population_stability_index(np.array([0.1, 0.9]), np.array([0.1, 0.1, 0.9, 0.9]),
                                       np.array([0.0, 0.5, 1.0]))
    assert abs(sized.psi) < 1e-12, (
        f"the same shape at twice the size scored {sized.psi:.6f} — divide each side by its own "
        "total before comparing")
    placed = population_stability_index(np.array([0.2, 0.7]), np.array([0.5, 1.0, 7.0, -3.0]),
                                        np.array([0.0, 0.5, 1.0]))
    assert list(placed.actual_pct) == [0.25, 0.75], (
        f"actual shares came back {list(placed.actual_pct)}, expected [0.25, 0.75]: a value ON "
        "an inner edge belongs to the bin above it, the last bin is closed on the right, and "
        "values beyond the outer edges are clipped into the end bins")
    emptied = population_stability_index(np.array([0.1, 0.9]), np.array([0.9, 0.9]),
                                         np.array([0.0, 0.5, 1.0]))
    assert np.isfinite(emptied.psi) and emptied.psi > 1.0, (
        f"a bin that emptied out scored {emptied.psi} — floor both shares before the log "
        "instead of letting log(0) erase the most important finding")
    assert abs(float(emptied.contributions.sum()) - emptied.psi) < 1e-12, \
        "contributions must sum to psi, or a finding cannot be traced back to a bin"
    for bad, why in ((np.array([0.0]), "fewer than two edges"),
                     (np.array([0.0, 0.5, 0.5, 1.0]), "a repeated edge")):
        try:
            population_stability_index(same, same, bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"{why} should raise ValueError")
    print("exercise 1 looks right")


_try("exercise 1", _check_psi)

## 3. Exercise 2 — `kl_divergence`

On fixed edges PSI is symmetric: swap the samples and it does not change. The Kullback-Leibler
divergence is not. `KL(current ‖ baseline) = Σ a_i · ln(a_i / e_i)` is, informally, the
surprise of someone who still believes the baseline and is shown today's traffic; it is large
where today puts mass the baseline hardly had. Add the two directions together and you get PSI back — PSI *is*
a symmetrised KL, which is why this lesson computes both on the same binned, floored shares.

One deliberate difference from a library `entropy` routine: the floored shares are NOT
renormalised to sum to one. That keeps `PSI = KL(a‖e) + KL(e‖a)` exact, bin for bin.

<details><summary>💡 Hint 1 — what to think about</summary>

Which sample sits in front of the logarithm is the whole difference between the two
directions. The weight is the CURRENT share; the ratio is current over baseline. A bin that
empties out on either side needs the same protection it got in PSI, and the identity with PSI
only holds if the floored shares are used in both the weight and the ratio — and if every
value, including one that sits exactly on an edge, lands in the bin exercise 1 put it in.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the edges exactly as exercise 1 does, bin both samples the same way, turn counts into
shares of each side's own total, floor both share vectors, then sum current share times the
natural log of current over baseline. Return a plain float, in nats.

</details>

In [ ]:
def kl_divergence(expected: np.ndarray, actual: np.ndarray, edges: np.ndarray,
                  floor: float = PSI_FLOOR) -> float:
    """KL(actual ‖ expected) in nats, on the same bins and floor as `population_stability_index`.

    `expected` is the baseline sample and `actual` the current one; both are binned on `edges`
    with the rules of exercise 1, turned into shares of their own totals, and floored at
    `floor`. The floored shares are used as they are — in the weight AND in the ratio, and
    without renormalising — so that `KL(a‖e) + KL(e‖a)` equals PSI exactly.

    Raise `ValueError` for edges that are not strictly increasing or hold fewer than two values.

    Example — the current window moved mass into the first bin:
        >>> base = np.array([0.1] * 5 + [0.9] * 5)       # shares 0.5, 0.5
        >>> now = np.array([0.1] * 9 + [0.9] * 1)        # shares 0.9, 0.1
        >>> round(kl_divergence(base, now, np.array([0.0, 0.5, 1.0])), 4)
        0.3681
    Returns:
        A float: sum over bins of a * ln(a / e), floored shares, natural log.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_kl() -> None:
    edges = np.array([0.0, 0.5, 1.0])
    base = np.array([0.1] * 5 + [0.9] * 5)
    now = np.array([0.1] * 9 + [0.9] * 1)
    forward = kl_divergence(base, now, edges)
    assert isinstance(forward, float), "return a plain float"
    assert abs(forward - (0.9 * np.log(0.9 / 0.5) + 0.1 * np.log(0.1 / 0.5))) < 1e-9, (
        f"KL(current‖baseline) should be 0.3681 nats, got {forward:.4f}. 0.5108 means you "
        "computed the other direction, KL(baseline‖current); 0.5310 means log base 2")
    backward = kl_divergence(now, base, edges)
    psi = (0.9 - 0.5) * np.log(0.9 / 0.5) + (0.1 - 0.5) * np.log(0.1 / 0.5)
    assert abs(forward + backward - psi) < 1e-9, \
        "the two directions must add up to PSI on the same shares"
    empty = kl_divergence(np.array([0.1, 0.9]), np.array([0.9, 0.9]), edges)
    assert np.isfinite(empty), \
        "a bin that emptied out must be floored, not turned into log(0)"
    assert abs(kl_divergence(base, base, edges)) < 1e-12, "identical samples diverge by 0"
    on_edge = kl_divergence(np.array([0.1, 0.1, 0.9]), np.array([0.5, 0.5]), edges)
    want = PSI_FLOOR * np.log(PSI_FLOOR / (2 / 3)) + 1.0 * np.log(1.0 / (1 / 3))
    assert abs(on_edge - want) < 1e-9, (
        f"two current values sitting ON the inner edge 0.5 gave {on_edge:.4f} nats, expected "
        f"{want:.4f}: bin exactly as exercise 1 does — a value on an inner edge belongs to the "
        "bin above it (numpy's searchsorted defaults to the other side)")
    try:
        kl_divergence(base, now, np.array([0.0, 0.0, 1.0]))
    except ValueError:
        pass
    else:
        raise AssertionError("a repeated edge should raise ValueError")
    print("exercise 2 looks right")


_try("exercise 2", _check_kl)

## 4. Exercise 3 — `fit_edges` and `drift_scores`

The weekly drift report scores the last `WINDOW_DAYS` of every signal against the baseline
quarter. Two design decisions carry most of the risk.

**The edges belong to the baseline.** Cut them once, on the baseline, and keep them. PSI is
symmetric only while the cut-off points are fixed in advance; cut them on whichever sample is
current and the number changes when the samples swap roles, so this week's figure is no
longer comparable with last week's. Section 12 measures the worse habit this invites.

**A categorical field gets one bin per category.** `currency` and `payment_terms_days` are
stored as codes. Quantile edges on codes collapse: repeated codes give repeated deciles, the
duplicates are merged, and two currencies can end up sharing a bin — a shift between them is
then invisible. The demo after the check measures it on this pipeline. `CATEGORIES` says which
signals are categorical and how many codes each has.

<details><summary>💡 Hint 1 — what to think about</summary>

For a categorical signal you want each code alone in its bin, with the edges strictly
increasing and every code strictly inside its bin. Which codes exist is a fact about the
field, not about the baseline: a currency the baseline quarter never saw can still turn up
next month, and it needs a bin of its own to show up in. For a continuous signal the lesson
already gives you the right edges. `drift_scores` should never cut an edge: it is handed them.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

`fit_edges`: for each signal in the baseline mapping, if it is in `categories` take its code
count from there and put an edge half-way below code 0, between every pair of neighbouring
codes and half-way above the last code; otherwise use `quantile_edges` on the baseline values. `drift_scores`: for every signal
in `edges`, score the current values against the baseline values with your two functions,
baseline first, and return a `Drift` per signal.

</details>

In [ ]:
def quantile_edges(x: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Bin edges at equally spaced quantiles of `x`, widened so nothing falls outside them.

    Given to you, unchanged from the model-risk programme's validation suite. The edges are cut
    ONCE, on the baseline sample, and then travel with it into monitoring.
    """
    edges = np.quantile(np.asarray(x, dtype=float), np.linspace(0.0, 1.0, n_bins + 1))
    edges = np.unique(edges)
    edges[0] = -np.inf
    edges[-1] = np.inf
    return edges


def fit_edges(baseline: Mapping[str, np.ndarray], categories: Mapping[str, int] = CATEGORIES,
              n_bins: int = N_BINS) -> dict[str, np.ndarray]:
    """Cut every signal's bin edges ONCE, on the baseline.

    * A signal named in `categories` holds integer codes `0 .. categories[name] - 1`. It gets
      one bin per code, with the edges half-way between codes: `[-0.5, 0.5, 1.5, ...]`.
    * Every other signal gets `quantile_edges(baseline[name], n_bins)`.

    Example:
        >>> e = fit_edges({"currency": np.array([0, 0, 1, 2]), "x": np.arange(100.0)})
        >>> e["currency"].tolist()
        [-0.5, 0.5, 1.5, 2.5]
        >>> e["x"].size - 1
        10
    Returns:
        A dict, signal name -> strictly increasing edges array, one entry per baseline signal.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def drift_scores(edges: Mapping[str, np.ndarray], baseline: Mapping[str, np.ndarray],
                 current: Mapping[str, np.ndarray]) -> dict[str, Drift]:
    """PSI and KL(current ‖ baseline) for every signal in `edges`, on THOSE edges.

    The edges were cut on the baseline by `fit_edges` and are never re-cut here. The baseline
    is the expected sample and `current` the actual one, for both statistics.

    Example:
        >>> base = {"x": np.array([0.1, 0.9])}
        >>> scores = drift_scores({"x": np.array([0.0, 0.5, 1.0])}, base, base)
        >>> (round(scores["x"].psi, 6), round(scores["x"].kl, 6))
        (0.0, 0.0)
    Returns:
        A dict, signal name -> Drift(psi, kl), with exactly the keys of `edges`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_edges_and_drift() -> None:
    rng = np.random.default_rng(3)
    base = {"currency": rng.choice(3, size=4000, p=[0.5, 0.3, 0.2]),
            "amount": rng.normal(3.0, 0.5, size=4000)}
    edges = fit_edges(base)
    assert set(edges) == set(base), f"one set of edges per baseline signal, got {sorted(edges)}"
    assert np.array_equal(edges["amount"], quantile_edges(base["amount"])), \
        "a continuous signal gets quantile_edges cut on the BASELINE values"
    assert edges["currency"].size == 4, (
        f"currency has 3 codes and needs 3 bins (4 edges), got {edges['currency'].size - 1} bins "
        "— quantile edges on codes merge categories")
    unseen = np.asarray(fit_edges({"currency": np.array([0, 0, 1])})["currency"], dtype=float)
    assert unseen.size == 4, (
        f"a baseline in which currency code 2 never occurs still needs a bin for it, got "
        f"{unseen.size - 1} bins — take the number of bins from `categories`, not from the codes "
        "that happened to turn up")
    moved = {"currency": rng.choice(3, size=2000, p=[0.5, 0.1, 0.4]),
             "amount": rng.normal(3.0, 0.5, size=2000)}
    scores = drift_scores(edges, base, moved)
    assert set(scores) == set(edges) and isinstance(scores["currency"], Drift), \
        "return a Drift(psi, kl) for every signal in edges"
    assert scores["currency"].psi > 0.1, (
        f"dollars turning into pounds scored PSI {scores['currency'].psi:.4f} — with quantile "
        "edges USD and GBP share one bin and the shift disappears")
    want = population_stability_index(base["amount"], moved["amount"], edges["amount"]).psi
    assert abs(scores["amount"].psi - want) < 1e-12, \
        "drift_scores must score on the edges it is given — never re-cut them on the window"
    want_kl = kl_divergence(base["currency"], moved["currency"], edges["currency"])
    assert abs(scores["currency"].kl - want_kl) < 1e-12, \
        "kl is KL(current ‖ baseline): baseline is the expected sample, current the actual one"
    print("exercise 3 looks right")


_try("exercise 3", _check_edges_and_drift)

Run this to see what the drift report says about each event. A signal family counts as
**moved** when its largest PSI exceeds a level measured, not chosen: the PSI that only
`DRIFT_FLAG_RATE` of simulated no-change windows reach. Each simulated window pairs a resampled
baseline quarter with an independent resampled four-week window, both drawn from the real
baseline, with edges cut on the resampled baseline — the same arithmetic a real window gets.

In [ ]:
class DriftLevels(NamedTuple):
    values: float          # a value signal "moved" when its PSI exceeds this
    confidence: float      # a confidence signal "moved" when its PSI exceeds this
    flag_rate: float       # share of simulated no-change windows that exceed each level
    n_draws: int


def drift_levels(log: PipelineLog, n_draws: int = DRIFT_DRAWS,
                 flag_rate: float = DRIFT_FLAG_RATE, seed: int = SEED + 2) -> DriftLevels:
    """The PSI a no-change window reaches, measured by resampling the baseline quarter."""
    rng = np.random.default_rng(seed)
    rows = BASELINE_DAYS * DOCS_PER_DAY
    peaks: dict[str, list[float]] = {"values": [], "confidence": []}
    for _ in range(n_draws):
        base = _signals_at(log, rng.integers(0, rows, size=rows))
        window = _signals_at(log, rng.integers(0, rows, size=WINDOW_DAYS * DOCS_PER_DAY))
        scores = drift_scores(fit_edges(base), base, window)
        peaks["values"].append(max(scores[s].psi for s in VALUE_SIGNALS))
        peaks["confidence"].append(max(scores[s].psi for s in CONFIDENCE_SIGNALS))
    q = 1.0 - flag_rate
    return DriftLevels(float(np.quantile(peaks["values"], q)),
                       float(np.quantile(peaks["confidence"], q)), flag_rate, n_draws)


# Four windows to read: one quiet, and one fully inside each event.
READ_WINDOWS = (("quiet", 132), ("half-year-end mix shift", 181), ("model regression", 244),
                ("vendor template change", 335))


def _show_what_drift_sees() -> None:
    global LEVELS
    baseline = window_signals(LOG, 0, BASELINE_DAYS - 1)
    edges = fit_edges(baseline)
    LEVELS = drift_levels(LOG)
    print(f"moved means PSI above {LEVELS.values:.4f} (values) or {LEVELS.confidence:.4f} "
          f"(confidence): {100 * LEVELS.flag_rate:.0f}% of {LEVELS.n_draws} no-change windows "
          "go higher.\n")
    print(f"{'window':26s}{'top value signal':>24s}{'PSI':>8s}{'top confidence signal':>34s}"
          f"{'PSI':>8s}  moved")
    moved_by = {}
    for name, end in READ_WINDOWS:
        scores = drift_scores(edges, baseline, window_signals(LOG, end - WINDOW_DAYS + 1, end))
        v = max(VALUE_SIGNALS, key=lambda s: scores[s].psi)
        c = max(CONFIDENCE_SIGNALS, key=lambda s: scores[s].psi)
        moved = [fam for fam, s, lvl in (("values", v, LEVELS.values),
                                        ("confidence", c, LEVELS.confidence))
                 if scores[s].psi > lvl]
        moved_by[name] = moved
        print(f"{name:26s}{v:>24s}{scores[v].psi:8.4f}{c:>34s}{scores[c].psi:8.4f}  "
              f"{' + '.join(moved) or 'nothing'}")
    end = READ_WINDOWS[-1][1]
    window = window_signals(LOG, end - WINDOW_DAYS + 1, end)
    scores = drift_scores(edges, baseline, window)
    s = max(CONFIDENCE_SIGNALS, key=lambda name: scores[name].psi)
    back = kl_divergence(window[s], baseline[s], edges[s])
    print(f"\nthe template window's {s}: PSI {scores[s].psi:.4f} = KL(current‖baseline) "
          f"{scores[s].kl:.4f} + KL(baseline‖current) {back:.4f}")
    end = READ_WINDOWS[1][1]
    mix = window_signals(LOG, end - WINDOW_DAYS + 1, end)
    naive = quantile_edges(baseline["currency"])
    merged = population_stability_index(baseline["currency"], mix["currency"], naive).psi
    right = population_stability_index(baseline["currency"], mix["currency"],
                                       edges["currency"]).psi
    print(f"quantile edges would give currency {naive.size - 1} bins for {len(CURRENCIES)} "
          f"currencies; the mix window's currency PSI would read {merged:.4f}, not {right:.4f}.")
    harm, mix = moved_by["model regression"], moved_by["half-year-end mix shift"]
    print("The regression moved no confidence signal: its wrong values carry a correct answer's")
    if harm == mix:
        print(f"confidence. It nudged the amounts just enough to read '{' + '.join(harm)}' moved,")
        print("exactly what the harmless mix shift reads. A drift report alone cannot tell the one")
        print("change that hurt from a change that did not.")
    else:
        print("confidence, and its nudge to the amounts reads differently from the mix shift only")
        print("in size. A drift report alone cannot tell you that quality moved.")


LEVELS = None
_try("what drift sees", _show_what_drift_sees, needs=("exercise 1", "exercise 2", "exercise 3"))

Now the obvious first monitor: page whenever any signal's PSI crosses `PSI_RULE_OF_THUMB`, the
conventional "significant change" band — a band that, as a study of PSI's statistical
properties points out, is used without reference to any error rate. Each page is traced to its
cause by replaying the same year with one event switched off (`COUNTERFACTUALS`).

In [ ]:
def rule_of_thumb_pager(log: PipelineLog) -> tuple[int, ...]:
    """Page on every weekly report in which any signal's PSI exceeds PSI_RULE_OF_THUMB."""
    baseline = window_signals(log, 0, BASELINE_DAYS - 1)
    edges = fit_edges(baseline)
    pages = []
    for day in range(MONITOR_START, N_DAYS, 7):
        window = window_signals(log, day - WINDOW_DAYS, day - 1)
        if max(population_stability_index(baseline[s], window[s], edges[s]).psi
               for s in SIGNALS) > PSI_RULE_OF_THUMB:
            pages.append(day)
    return tuple(pages)


def caused_pages(pager: Callable[[PipelineLog], Sequence[int]]) -> dict[str, tuple[int, ...]]:
    """Which of a pager's pages each event caused: pages on LOG that vanish when the event is
    switched off. What no event caused is 'unexplained' — the false pages."""
    full = set(pager(LOG))
    out = {e.kind: tuple(sorted(full - set(pager(COUNTERFACTUALS[e.kind])))) for e in EVENTS}
    explained = set().union(*(set(v) for v in out.values()))
    out["unexplained"] = tuple(sorted(full - explained))
    return out


def _describe(caused: Mapping[str, tuple[int, ...]]) -> None:
    for e in EVENTS:
        days = caused[e.kind]
        print(f"  {e.name:26s} {len(days):3d} pages" + (f", first on day {days[0]}" if days else ""))
    print(f"  {'caused by nothing':26s} {len(caused['unexplained']):3d} pages")


def _show_rule_of_thumb() -> None:
    caused = caused_pages(rule_of_thumb_pager)
    print(f"page when any PSI > {PSI_RULE_OF_THUMB}:")
    _describe(caused)
    harmless = [e.name for e in EVENTS if e.kind != "regression" and caused[e.kind]]
    n_harmless = sum(e.kind != "regression" for e in EVENTS)
    print(f"It paged for {len(harmless)} of the {n_harmless} harmless changes and "
          f"{len(caused['regression'])} times for the regression.")


_try("rule of thumb", _show_rule_of_thumb, needs=("exercise 1", "exercise 2", "exercise 3"))

## 5. Exercise 4 — `daily_defect_rate`

Drift tells you WHERE the traffic moved. It cannot tell you whether the extractor got worse,
because a model can be confidently wrong on traffic that looks exactly like the baseline. For
that you need labels, and the only labels a live pipeline has are the audit's.

The quality proxy is the day's **audited defect rate**: of the cells a reviewer checked that
day, the share labelled `miss`, `spurious` or `wrong_value` — module 1's three defect classes.
A `true_negative` is a cell the extractor got right by staying silent, so it belongs in the
denominator. A cell nobody reviewed carries `NOT_AUDITED` and belongs nowhere.

Why a RANDOM audit, and not module 1's confidence-ordered review queue? That queue only ever
looks at the least confident cells. A regression that is confidently wrong never reaches it.

<details><summary>💡 Hint 1 — what to think about</summary>

Three decisions: which cells count at all (only reviewed ones), which labels are defects (three
of the five), and what the denominator is (cells, not documents, and every reviewed cell,
including the ones where the right answer was nothing). A day with nothing reviewed has no
rate at all, and pretending it is zero would hide a broken audit.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Map the defect label names to their codes. Per document, count its reviewed cells and its
defect cells; then add those counts up per day, for every day from 0 to `n_days - 1` whether
or not it has documents. If any day has zero reviewed cells, raise ValueError. Otherwise
divide defects by reviewed cells.

</details>

In [ ]:
def daily_defect_rate(day: np.ndarray, verdict: np.ndarray, n_days: int) -> np.ndarray:
    """The audited defect rate of each day `0 .. n_days - 1`.

    `day[i]` is document i's day; `verdict[i, j]` is the code in `ERROR_LABELS` a reviewer gave
    field j of document i, or `NOT_AUDITED` if nobody reviewed it. A day's rate is its reviewed
    cells labelled one of `DEFECT_LABELS`, over ALL its reviewed cells — true negatives
    included, unreviewed cells excluded from both. A day with no reviewed cell has no rate:
    raise `ValueError`.

    Example — one day, two documents, one of them reviewed:
        >>> v = np.array([[0, 3, 4], [-1, -1, -1]])     # correct, wrong_value, true_negative
        >>> daily_defect_rate(np.array([0, 0]), v, 1).round(4).tolist()
        [0.3333]
    Returns:
        A float array of length `n_days`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_defect_rate() -> None:
    code = {label: i for i, label in enumerate(ERROR_LABELS)}
    v = np.array([[code["correct"], code["wrong_value"], code["true_negative"]],
                  [NOT_AUDITED, NOT_AUDITED, NOT_AUDITED],
                  [code["miss"], code["spurious"], code["correct"]]])
    got = daily_defect_rate(np.array([0, 0, 1]), v, 2)
    assert got.shape == (2,), f"one rate per day, got shape {got.shape}"
    assert abs(got[0] - 1 / 3) < 1e-12, (
        f"day 0 should be 1/3, got {got[0]:.4f}: 1/6 means unreviewed cells went into the "
        "denominator, 1/2 means the true negative was left out of it")
    assert abs(got[1] - 2 / 3) < 1e-12, (
        f"day 1 should be 2/3, got {got[1]:.4f}: miss and spurious are defects too, not only "
        "wrong_value")
    try:
        daily_defect_rate(np.array([0]), np.array([[code["correct"]]]), 2)
    except ValueError:
        pass
    else:
        raise AssertionError("day 1 has no reviewed cell: raise ValueError, do not report 0.0")
    print("exercise 4 looks right")


_try("exercise 4", _check_defect_rate)

The audit is a sample, so the proxy is noisy. Run this to compare it, four weeks at a time,
with the rate over EVERY cell — the column production never has.

In [ ]:
def _show_audit_against_truth() -> None:
    audited = daily_defect_rate(LOG.day, LOG.verdict, N_DAYS)
    truth = daily_defect_rate(LOG.day, LOG.truth, N_DAYS)
    print(f"{'days':>9s}{'audited':>10s}{'every cell':>12s}   daily audited sd")
    for b in range(0, N_DAYS, WINDOW_DAYS):
        s = slice(b, b + WINDOW_DAYS)
        print(f"{b:4d}-{b + WINDOW_DAYS - 1:<4d}{audited[s].mean():10.4f}{truth[s].mean():12.4f}"
              f"   {audited[s].std(ddof=1):.4f}")
    reg = next(e for e in EVENTS if e.kind == "regression")
    rise = truth[reg.start:reg.end].mean() - truth[:BASELINE_DAYS].mean()
    noise = audited[:BASELINE_DAYS].std(ddof=1)
    print(f"\nthe regression raised the rate over every cell by {rise:.4f}; one audited day "
          f"wobbles with an sd of {noise:.4f}.")
    print("A single day's noise is the size of the whole regression, so a one-day limit either")
    print("pages for noise or waits for luck. The next exercise accumulates evidence instead.")


_try("audit against truth", _show_audit_against_truth, needs=("exercise 4",))

## 6. Exercise 5 — `cusum`

A CUSUM chart accumulates small, persistent excesses that no single day shows, which is why
NIST's handbook finds it better than a one-day (Shewhart) limit at detecting shifts of two
sigma or less. On the standardised proxy `z_t = (x_t − μ0) / σ0`, with `μ0` and `σ0` from the
baseline quarter, the upper chart is the tabular form NIST's handbook gives:

`S_t = max(0, S_{t−1} + z_t − k)`, starting from `S_{−1} = 0`,

and it pages on any day `S_t` **exceeds** `h`. `k` is the reference value: choose it as half
the shift you must catch, here `MIN_SHIFT / (2 σ0)`. After a page the chart **resets** to zero
— somebody has been told, and the next page must be earned by new evidence. Only a rise in the
defect rate is watched: an extractor that gets better does not need waking up for.

<details><summary>💡 Hint 1 — what to think about</summary>

Three details carry the marks: the floor at zero (a good week must not bank credit against a
bad one), the strict comparison with `h`, and the reset. Decide what the statistic records on
a page day — the value that crossed, so the chart shows the crossing — and where the next day
starts from.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Keep a running value starting at zero. For each day in order, add the day's z minus k and
floor the result at zero; record it. If it is strictly greater than h, record the day's index
as a page and set the running value back to zero before the next day. Return the recorded
values as an array and the page indices as a tuple of ints.

</details>

In [ ]:
def cusum(z: np.ndarray, k: float, h: float) -> CusumResult:
    """Upper one-sided CUSUM with reset, on a standardised daily series `z`.

    `S_t = max(0, S_{t-1} + z_t - k)` with `S_{-1} = 0`. Day t pages when `S_t > h` (strictly).
    `statistic[t]` is `S_t` as computed on day t — the crossing value on a page day. After a
    page, the next day starts again from `S = 0`.

    Example — a step up of 2 sigma on day 2:
        >>> r = cusum(np.array([0.0, 0.0, 2.0, 2.0, 2.0]), k=0.5, h=2.0)
        >>> r.statistic.tolist(), r.pages
        ([0.0, 0.0, 1.5, 3.0, 1.5], (3,))
    Returns:
        A CusumResult(statistic, pages): a float array as long as `z`, and a tuple of the
        integer day indices that paged.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_cusum() -> None:
    r = cusum(np.array([0.0, 0.0, 2.0, 2.0, 2.0]), k=0.5, h=2.0)
    assert isinstance(r, CusumResult), "return a CusumResult(statistic, pages)"
    assert np.allclose(r.statistic, [0.0, 0.0, 1.5, 3.0, 1.5]), (
        f"got statistic {np.round(r.statistic, 3).tolist()}, expected [0, 0, 1.5, 3, 1.5]: "
        "subtract k every day, floor at zero, record the crossing value, then reset")
    assert r.pages == (3,), f"expected a page on day 3 only, got {r.pages}"
    low = cusum(np.array([-3.0, 1.0]), k=0.0, h=5.0)
    assert np.allclose(low.statistic, [0.0, 1.0]), \
        "floor S at zero every day: a good day must not bank credit against a bad one"
    edge = cusum(np.array([2.5]), k=0.5, h=2.0)
    assert edge.pages == (), "S equal to h is not a page: the chart pages when S EXCEEDS h"
    flat = cusum(np.array([3.0, 3.0, 3.0, 3.0]), k=0.0, h=5.0)
    assert flat.pages == (1, 3), (
        f"expected pages on days 1 and 3, got {flat.pages} — after a page the chart restarts "
        "from zero, so the next page needs new evidence")
    print("exercise 5 looks right")


_try("exercise 5", _check_cusum)

Run your CUSUM over the replay year with the handbook's rule-of-thumb `h`. It finds the
regression. What nobody can tell you yet is how often this `h` pages for nothing.

In [ ]:
def _baseline_calibration() -> tuple[np.ndarray, float, float, float]:
    rates = daily_defect_rate(LOG.day, LOG.verdict, N_DAYS)
    mu0, sigma0 = float(rates[:BASELINE_DAYS].mean()), float(rates[:BASELINE_DAYS].std(ddof=1))
    return rates, mu0, sigma0, MIN_SHIFT / (2 * sigma0)


def _show_round_number_cusum() -> None:
    rates, mu0, sigma0, k = _baseline_calibration()
    h = ROUND_THRESHOLDS[0]
    z = (rates - mu0) / sigma0
    pages = [MONITOR_START + t for t in cusum(z[MONITOR_START:], k, h).pages]
    print(f"baseline: mu0 = {mu0:.4f}, sigma0 = {sigma0:.4f} per day; k = {k:.3f} sigma "
          f"(half of MIN_SHIFT = {MIN_SHIFT})")
    print(f"h = {h}: pages on days {pages}")
    print("What is this h's false-page rate? Nothing on this page says. Exercise 6 measures it.")


_try("round-number cusum", _show_round_number_cusum, needs=("exercise 4", "exercise 5"))

## 7. Exercise 6 — a threshold that states its false-page rate

A false page is a page on a day nothing changed. You cannot count them on the real year —
you do not know which days those are — so you count them on **simulated no-change years**.
`null_years()` builds each one by resampling the baseline quarter's daily rates with
replacement: the only thing you know for certain about "no change" is the quarter you
calibrated on. Standardised with the same `μ0` and `σ0`, and run through your CUSUM, they say
what each `h` costs in false pages per quarter.

The rule is then mechanical. `FALSE_PAGE_BUDGET` is what the on-call rota agreed to absorb;
take the **smallest** `h` in the grid whose measured rate is **at most** the budget — the most
sensitive chart the rota will tolerate — and carry the rate with it. One honest limit: a
resampled year never holds a day worse than the worst baseline day, so its tail is thinner
than reality's. Section 9 checks the budget again on the replay's own quiet stretches.

<details><summary>💡 Hint 1 — what to think about</summary>

The unit is pages per quarter, and it must survive a no-change sample of any length: count
EVERY page, not the years that had one, and convert days into quarters with `QUARTER_DAYS`.
For the choice, ask which end of the grid is the sensitive one, whether "at most" includes
the budget itself, and what to do if no `h` in the grid is quiet enough.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

`false_pages_per_quarter`: run `cusum` on each row, add up the pages, and divide by the total
number of simulated days over `QUARTER_DAYS`. `choose_threshold`: walk the grid from the
smallest h upward, measure each rate, and return the first `Threshold` whose rate is within
budget; if none is, raise ValueError rather than hand back a threshold that breaks the budget.

</details>

In [ ]:
def null_years(baseline_rates: np.ndarray, n_years: int = NULL_YEARS, n_days: int = N_DAYS,
               seed: int = SEED + 1) -> np.ndarray:
    """Simulated no-change years: each day drawn with replacement from the baseline's days."""
    rng = np.random.default_rng(seed)
    return rng.choice(np.asarray(baseline_rates, dtype=float), size=(n_years, n_days))


def false_pages_per_quarter(null_z: np.ndarray, k: float, h: float) -> float:
    """Mean number of CUSUM pages per quarter over simulated no-change years.

    `null_z` holds one standardised simulated year per row (any number of rows, any number of
    days). Every page counts — a year that pages three times contributes three — and the total
    is divided by the number of simulated quarters: rows x days / QUARTER_DAYS.

    Example — two 91-day years, one with a single enormous day:
        >>> null = np.zeros((2, QUARTER_DAYS)); null[0, 10] = 9.0
        >>> false_pages_per_quarter(null, k=0.5, h=4.0)
        0.5
    Returns:
        A float, false pages per quarter.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def choose_threshold(null_z: np.ndarray, k: float, grid: Sequence[float],
                     budget: float) -> Threshold:
    """The smallest h in `grid` whose false-page rate on `null_z` is at most `budget`.

    Smallest, because a lower h pages sooner on a real shift: this is the most sensitive
    threshold the budget allows. The grid may arrive in any order. If no h meets the budget,
    raise `ValueError` — a threshold that breaks its own budget is not a threshold.

    Example:
        >>> null = np.zeros((2, QUARTER_DAYS)); null[0, 10] = 9.0
        >>> choose_threshold(null, k=0.5, grid=[10.0, 4.0], budget=0.5)
        Threshold(h=4.0, false_pages_per_quarter=0.5, budget=0.5)
    Returns:
        A Threshold(h, false_pages_per_quarter, budget) — h together with its measured rate.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_threshold() -> None:
    null = np.zeros((2, QUARTER_DAYS))
    null[0, 10] = 9.0
    rate = false_pages_per_quarter(null, k=0.5, h=4.0)
    assert abs(rate - 0.5) < 1e-12, (
        f"one page over two simulated quarters is 0.5 per quarter, got {rate} — divide every "
        "page by rows x days / QUARTER_DAYS")
    twice = np.zeros((1, 2 * QUARTER_DAYS))
    twice[0, 5] = twice[0, 100] = 9.0
    rate = false_pages_per_quarter(twice, k=0.5, h=4.0)
    assert abs(rate - 1.0) < 1e-12, (
        f"two pages in one two-quarter year is 1.0 per quarter, got {rate} — count every page, "
        "not the years that paged, and convert the days you were given into quarters")
    chosen = choose_threshold(null, k=0.5, grid=[10.0, 2.0, 4.0], budget=0.5)
    assert isinstance(chosen, Threshold), "return a Threshold(h, false_pages_per_quarter, budget)"
    assert chosen.h == 2.0 and chosen.false_pages_per_quarter == 0.5, (
        f"got {chosen}: the smallest h whose rate is AT MOST the budget is 2.0 — sort the grid, "
        "and 'at most' includes the budget itself")
    try:
        choose_threshold(null, k=0.5, grid=[1.0, 2.0], budget=0.1)
    except ValueError:
        pass
    else:
        raise AssertionError("no h meets a budget of 0.1 here: raise ValueError")
    print("exercise 6 looks right")


_try("exercise 6", _check_threshold)

Run this for the table a rota owner signs: each `h` beside what it costs in false pages and
what it buys in speed. Speed is measured on the same simulated years with a rise of exactly
`MIN_SHIFT` added from their first day, and priced in documents the broken model reads first.

In [ ]:
def _days_to_page(null_z: np.ndarray, k: float, h: float) -> float:
    firsts = [(cusum(row, k, h).pages or (len(row) - 1,))[0] + 1 for row in null_z]
    return float(np.mean(firsts))


def _show_threshold_table() -> None:
    global THRESHOLD
    rates, mu0, sigma0, k = _baseline_calibration()
    null_z = (null_years(rates[:BASELINE_DAYS]) - mu0) / sigma0
    THRESHOLD = choose_threshold(null_z, k, H_GRID, FALSE_PAGE_BUDGET)
    shifted = null_z + MIN_SHIFT / sigma0
    below = max((h for h in H_GRID if h < THRESHOLD.h), default=THRESHOLD.h)
    print(f"{NULL_YEARS} simulated no-change years; budget {FALSE_PAGE_BUDGET} false pages per "
          f"quarter; k = {k:.3f} sigma\n")
    print(f"{'threshold':34s}{'false pages / quarter':>23s}{'days to page MIN_SHIFT':>24s}"
          f"{'documents first':>17s}")
    rows = [(f"h = {below} (one grid step lower)", below),
            (f"h = {THRESHOLD.h} (chosen by the budget)", THRESHOLD.h)]
    rows += [(f"h = {h} (handbook rule of thumb)", h) for h in ROUND_THRESHOLDS]
    for name, h in rows:
        fp, days = false_pages_per_quarter(null_z, k, h), _days_to_page(shifted, k, h)
        flag = "" if fp <= FALSE_PAGE_BUDGET else "  over budget"
        print(f"{name:34s}{fp:23.3f}{days:24.1f}{days * DOCS_PER_DAY:17,.0f}{flag}")
    print("\n'documents first' is how many documents the broken model reads before the page.")
    print("The round numbers are not wrong for being round. They are wrong because, until this")
    print("table, nobody could say what they cost — in false pages or in documents.")


THRESHOLD = None
_try("threshold table", _show_threshold_table,
     needs=("exercise 4", "exercise 5", "exercise 6"))

## 8. Exercise 7 — `triage`

Every day the monitor now holds three facts: did the quality CUSUM page, did any value signal
move, did any confidence signal move. `triage` turns them into a label and a decision, and the
decision is the one the whole lesson has been measuring its way to: **page if and only if
quality moved**. Drift never pages. It explains.

| quality moved | values moved | confidence moved | label |
|---|---|---|---|
| yes | any | any | `regression` — page |
| no | no | yes | `template change` — the extractor is reading layouts it had not seen |
| no | yes | no | `mix shift` — different documents, the same extractor |
| no | yes | yes | `template change and mix shift` |
| no | no | no | `no change` |

<details><summary>💡 Hint 1 — what to think about</summary>

The trap is precedence. When quality moved AND a distribution moved, it is tempting to say
the drift explains the quality change and stay quiet. It does not: a template change that
breaks extraction is a regression like any other, and the customer sees the wrong amount
either way.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Test quality first and return the regression label with `page=True`. Otherwise the page
flag is False, and the label is chosen from the two drift flags exactly as the table reads.

</details>

In [ ]:
def triage(quality_moved: bool, values_moved: bool, confidence_moved: bool) -> Triage:
    """Name what the monitor sees, and decide whether to page.

    Page if and only if `quality_moved`. The label follows the table above, with quality
    taking precedence over both drift flags.

    Examples:
        >>> triage(True, True, False)
        Triage(label='regression', page=True)
        >>> triage(False, False, True)
        Triage(label='template change', page=False)
    Returns:
        A Triage(label, page) with `label` one of TRIAGE_LABELS.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_triage() -> None:
    got = triage(True, False, False)
    assert isinstance(got, Triage) and got == ("regression", True), (
        f"quality moved and nothing else did: got {got}. A confidently wrong model moves no "
        "distribution at all; it must still page")
    for v, c in ((True, False), (False, True), (True, True)):
        got = triage(True, v, c)
        assert got.page, (
            f"quality moved while values_moved={v}, confidence_moved={c}, and triage did not "
            "page — drift does not explain a quality change away")
        got = triage(False, v, c)
        assert not got.page, (
            f"only drift moved (values_moved={v}, confidence_moved={c}) and triage paged — drift "
            "never pages; it says where to look")
    assert triage(False, False, True).label == "template change", \
        "confidence moved, values did not: the extractor is reading an unfamiliar layout"
    assert triage(False, True, False).label == "mix shift", \
        "values moved, confidence did not: different documents, the same extractor"
    both = triage(False, True, True).label
    assert both == "template change and mix shift", (
        f"values AND confidence moved, quality did not: got {both!r}. Test the both-moved case "
        "before either flag alone, or it is swallowed by whichever single-flag branch comes first")
    quiet = triage(False, False, False).label
    assert quiet == "no change", (
        f"nothing moved and triage said {quiet!r}: with no flag set the label is 'no change'")
    print("exercise 7 looks right")


_try("exercise 7", _check_triage)

## 9. The replay

Everything is now in place. `build_monitor` calibrates on the baseline quarter and nowhere
else: edges cut once, drift levels, `μ0` and `σ0`, `k`, and the threshold with its rate.
`replay` then walks the monitored days. Every seventh day it files a drift report on the
previous `WINDOW_DAYS`; every day it updates the CUSUM and asks `triage` whether to page.
`caused_pages` replays the same year with each event switched off, so every page is traced to
the event that caused it, or to nothing.

In [ ]:
class Monitor(NamedTuple):
    edges: dict              # cut ONCE, on the baseline quarter
    baseline: dict           # baseline signal values
    levels: DriftLevels      # when a family counts as moved, with its flag rate
    mu0: float               # baseline audited defect rate, mean per day
    sigma0: float            # and its day-to-day sd
    k: float                 # CUSUM reference value, sigma units: half of MIN_SHIFT
    threshold: Threshold     # h, with its measured false pages per quarter


class Replay(NamedTuple):
    pages: tuple             # days somebody was paged
    labels: dict             # day -> triage label, every monitored day
    statistic: np.ndarray    # the CUSUM statistic of every monitored day
    rates: np.ndarray        # the audited defect rate of every day of the year


def build_monitor(log: PipelineLog) -> Monitor:
    """Calibrate every part of the monitor on the baseline quarter of `log`, and nothing else."""
    baseline = window_signals(log, 0, BASELINE_DAYS - 1)
    edges = fit_edges(baseline)
    rates = daily_defect_rate(log.day, log.verdict, N_DAYS)
    mu0, sigma0 = float(rates[:BASELINE_DAYS].mean()), float(rates[:BASELINE_DAYS].std(ddof=1))
    k = MIN_SHIFT / (2 * sigma0)
    null_z = (null_years(rates[:BASELINE_DAYS]) - mu0) / sigma0
    threshold = choose_threshold(null_z, k, H_GRID, FALSE_PAGE_BUDGET)
    return Monitor(edges, baseline, drift_levels(log), mu0, sigma0, k, threshold)


def replay(log: PipelineLog, monitor: Monitor) -> Replay:
    """Run the monitor over the monitored days of `log`, exactly as it would have run live."""
    rates = daily_defect_rate(log.day, log.verdict, N_DAYS)
    z = (rates - monitor.mu0) / monitor.sigma0
    chart = cusum(z[MONITOR_START:], monitor.k, monitor.threshold.h)
    alarms = {MONITOR_START + int(t) for t in chart.pages}
    flags, labels, pages = (False, False), {}, []
    for day in range(MONITOR_START, N_DAYS):
        if (day - MONITOR_START) % 7 == 0:       # the weekly report, on the previous window
            scores = drift_scores(monitor.edges, monitor.baseline,
                                  window_signals(log, day - WINDOW_DAYS, day - 1))
            flags = (max(scores[s].psi for s in VALUE_SIGNALS) > monitor.levels.values,
                     max(scores[s].psi for s in CONFIDENCE_SIGNALS) > monitor.levels.confidence)
        verdict = triage(day in alarms, *flags)
        labels[day] = verdict.label
        if verdict.page:
            pages.append(day)
    return Replay(tuple(pages), labels, chart.statistic, rates)


MONITOR = REPLAY = CAUSED = None
_ALL = tuple(_EXERCISES)


def _run_the_replay() -> None:
    global MONITOR, REPLAY, CAUSED
    MONITOR = build_monitor(LOG)
    REPLAY = replay(LOG, MONITOR)
    CAUSED = caused = caused_pages(lambda log: replay(log, MONITOR).pages)
    reg = next(e for e in EVENTS if e.kind == "regression")
    print(f"threshold h = {MONITOR.threshold.h} sigma, measured at "
          f"{MONITOR.threshold.false_pages_per_quarter:.3f} false pages per quarter "
          f"(budget {MONITOR.threshold.budget})\n")
    print("what triage called each day of each event (label: days):")
    for e in EVENTS:
        seen = [REPLAY.labels[d] for d in range(max(e.start, MONITOR_START), e.end)]
        tally = {label: seen.count(label) for label in TRIAGE_LABELS if seen.count(label)}
        print(f"  during the {e.name:26s} {tally}")
    misnamed = sum(REPLAY.labels[d] == "mix shift" for d in range(reg.start, reg.end))
    if misnamed:
        print(f"  On {misnamed} regression days between pages the label was 'mix shift': wrong "
              "amounts nudge the\n  amount distribution, and drift alone would have misnamed "
              "the harm.")
    print("\npages, traced to their cause:")
    _describe(caused)
    quiet = sum(1 for d in range(MONITOR_START, N_DAYS)
                if not any(e.start <= d < e.end for e in EVENTS))
    print(f"\n{len(caused['unexplained'])} false pages in the {quiet} event-free monitored days; "
          f"the budget allowed about {FALSE_PAGE_BUDGET * quiet / QUARTER_DAYS:.1f}.")
    hits = [d for d in caused["regression"] if reg.start <= d < reg.end]
    if hits:
        print(f"the regression went live on day {reg.start} and paged on day {hits[0]}: "
              f"{hits[0] - reg.start + 1} days and {(hits[0] - reg.start + 1) * DOCS_PER_DAY:,} "
              f"documents before anyone was told,\nand {reg.end - hits[0] - 1} days before the "
              "customer complaint that rolled it back.")
    else:
        print("the regression was never caught: the customer complaint found it first.")
    noisy = [e.name for e in EVENTS if e.kind != "regression" and caused[e.kind]]
    print("paged for a harmless change: " + (", ".join(noisy) if noisy else
                                             "neither — both were seen, labelled, and let be."))


_try("replay", _run_the_replay, needs=_ALL)

The CUSUM chart of the replay year. Top: the daily audited defect rate against the rate over
every cell. Bottom: the CUSUM statistic, the chosen `h`, and the pages. The shaded bands are
the three events.

In [ ]:
def _plot_the_year() -> None:
    truth = daily_defect_rate(LOG.day, LOG.truth, N_DAYS)
    days = np.arange(N_DAYS)
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    for ax in (top, bottom):
        for e, colour in zip(EVENTS, ("tab:blue", "tab:red", "tab:green")):
            ax.axvspan(e.start, e.end, color=colour, alpha=0.12, label=e.name)
        ax.axvline(MONITOR_START, color="grey", lw=0.8, ls=":")
    top.plot(days, REPLAY.rates, ".", ms=3, color="0.35", label="audited, per day")
    top.plot(days, truth, "-", lw=0.8, color="black", label="every cell (production never sees)")
    top.set_ylabel("defect rate")
    top.legend(loc="upper left", fontsize=7, ncol=3)
    watched = np.arange(MONITOR_START, N_DAYS)
    bottom.plot(watched, REPLAY.statistic, color="tab:purple", lw=1.0, label="CUSUM S_t")
    bottom.axhline(MONITOR.threshold.h, color="black", lw=0.8, ls="--",
                   label=f"h = {MONITOR.threshold.h}, "
                         f"{MONITOR.threshold.false_pages_per_quarter:.2f} false pages / quarter")
    bottom.plot(REPLAY.pages, [MONITOR.threshold.h] * len(REPLAY.pages), "v", color="tab:red",
                label="page")
    bottom.set_xlabel("day of the replay year")
    bottom.set_ylabel("S_t (sigma)")
    bottom.legend(loc="upper left", fontsize=7)
    fig.tight_layout()
    _show(fig)
    print(f"{len(REPLAY.pages)} pages, all between day {min(REPLAY.pages)} and day "
          f"{max(REPLAY.pages)}." if REPLAY.pages else "no pages")


_try("cusum chart", _plot_the_year, needs=_ALL)

## 10. Three monitors, one year

The same traffic, three pagers, every page traced to its cause. The first is the rule-of-
thumb PSI pager from section 4. The second is the cheapest quality proxy there is, one that
needs no reviewer at all: one minus the day's mean confidence, with its CUSUM held to the
same budget by the same machinery you just wrote. The third is yours.

In [ ]:
def confidence_proxy_pager(log: PipelineLog) -> tuple[int, ...]:
    """A label-free monitor: CUSUM on 1 - mean confidence, threshold from the same budget."""
    conf = np.column_stack([log.signals[s] for s in CONFIDENCE_SIGNALS]).mean(axis=1)
    proxy = 1.0 - np.bincount(log.day, weights=conf, minlength=N_DAYS) / DOCS_PER_DAY
    mu, sd = proxy[:BASELINE_DAYS].mean(), proxy[:BASELINE_DAYS].std(ddof=1)
    null_z = (null_years(proxy[:BASELINE_DAYS]) - mu) / sd
    h = choose_threshold(null_z, MONITOR.k, H_GRID, FALSE_PAGE_BUDGET).h
    chart = cusum(((proxy - mu) / sd)[MONITOR_START:], MONITOR.k, h)
    return tuple(MONITOR_START + int(t) for t in chart.pages)


def _show_three_monitors() -> None:
    pagers = (("PSI above the rule of thumb", rule_of_thumb_pager),
              ("CUSUM on 1 - mean confidence", confidence_proxy_pager),
              ("CUSUM on the audit + triage", lambda log: replay(log, MONITOR).pages))
    reg = next(e for e in EVENTS if e.kind == "regression")
    print(f"{'monitor':32s}{'mix shift':>11s}{'template':>10s}{'regression':>12s}"
          f"{'nothing':>9s}   regression caught on")
    blind = []
    for name, pager in pagers:
        c = CAUSED if pager is pagers[-1][1] else caused_pages(pager)
        first = [d for d in c["regression"] if reg.start <= d < reg.end]
        print(f"{name:32s}{len(c['mix']):11d}{len(c['template']):10d}{len(c['regression']):12d}"
              f"{len(c['unexplained']):9d}   " + (f"day {first[0]}" if first else "never"))
        if not first and (c["mix"] or c["template"]):
            blind.append(name)
    print("\nColumns count pages CAUSED by each event: pages that vanish when it is switched off.")
    if blind:
        print(f"Paged for a harmless change and never for the regression: {'; '.join(blind)}.")


_try("three monitors", _show_three_monitors, needs=_ALL)

## 11. The artefact: a monitor card

A threshold nobody can defend is a threshold somebody will quietly raise after the third
night of false pages. The card below is what this lesson ships: every threshold on it states
the rate at which it fires when nothing is wrong, and how that rate was measured.

In [ ]:
def monitor_card(monitor: Monitor, caused: Mapping[str, tuple]) -> str:
    """The monitor as a reviewer reads it. Every figure is computed from `monitor` and from the
    replay's pages traced to their causes (`caused_pages`)."""
    reg = next(e for e in EVENTS if e.kind == "regression")
    hits = [d for d in caused["regression"] if reg.start <= d < reg.end]
    lines = [
        "MONITOR CARD — remittance-advice extraction, synthetic replay year",
        f"pager signal   audited defect rate: {AUDIT_DOCS} random documents a day, every cell",
        f"baseline       days 0-{BASELINE_DAYS - 1}: mean {monitor.mu0:.4f}, "
        f"sd {monitor.sigma0:.4f} per day",
        f"chart          upper CUSUM with reset, k = {monitor.k:.3f} sigma "
        f"(half of a {MIN_SHIFT} rise), h = {monitor.threshold.h} sigma",
        f"false pages    {monitor.threshold.false_pages_per_quarter:.3f} per quarter, measured "
        f"on {NULL_YEARS} simulated no-change years (budget {monitor.threshold.budget})",
        f"drift report   weekly, last {WINDOW_DAYS} days vs edges cut once on the baseline; "
        "PSI and KL(current‖baseline); never pages",
        f"drift levels   values {monitor.levels.values:.4f}, confidence "
        f"{monitor.levels.confidence:.4f} PSI; each exceeded by "
        f"{100 * monitor.levels.flag_rate:.0f}% of {monitor.levels.n_draws} no-change windows",
        f"replay         regression paged on day {hits[0] if hits else 'never'}; pages caused "
        f"by the mix shift {len(caused['mix'])}, by the template change "
        f"{len(caused['template'])}, by nothing {len(caused['unexplained'])}",
    ]
    return "\n".join(lines)


_try("monitor card", lambda: print(monitor_card(MONITOR, CAUSED)), needs=_ALL)

## 12. Common mistakes

- **Paging on drift.** PSI measures how far the traffic moved, not whether the extractor got
  worse. Section 10 counts what that costs: pages for both harmless changes, none for the harm.
- **A label-free quality proxy.** Mean confidence is cheap and it is a drift statistic in
  disguise. A confidently wrong model does not move it.
- **Auditing by confidence.** Module 1's review queue spends its budget on the least confident
  cells, which is right for fixing errors and blind to a confidently wrong regression. The
  audit that feeds the pager must be random.
- **Re-cutting the edges on the current window.** The number then depends on which sample the
  cut-offs came from, and one week's figure stops being comparable with the next. Cut once, on
  the baseline.
- **Quantile edges on a categorical field.** They merge categories, and a shift between two
  merged categories is invisible.
- **A round-number threshold.** `h = 4` may be fine. Until it is measured on no-change years
  nobody knows its false-page rate or how many documents it lets through first.
- **Letting the drift explain the page away.** A template change that breaks extraction is a
  regression. Quality moved: page.
- **A rolling baseline.** Score each window against the one before it and a persistent change
  is the baseline a month later. Everything in `build_monitor` reads the baseline quarter and
  nothing else; re-baseline only as a deliberate, reviewed act.

The last mistake is worth seeing rather than believing. Run this: the weekly confidence report
through the template change, scored against the frozen baseline quarter and against a rolling
baseline — the previous four weeks, with edges re-cut on them each time.

In [ ]:
def _show_rolling_baseline() -> None:
    frozen = window_signals(LOG, 0, BASELINE_DAYS - 1)
    frozen_edges = fit_edges(frozen)
    tpl = next(e for e in EVENTS if e.kind == "template")
    print(f"{'report day':>11s}{'frozen baseline':>17s}{'rolling baseline':>18s}   max confidence PSI")
    for day in range(tpl.start, N_DAYS, 7):
        now = window_signals(LOG, day - WINDOW_DAYS, day - 1)
        before = window_signals(LOG, day - 2 * WINDOW_DAYS, day - WINDOW_DAYS - 1)
        before_edges = fit_edges(before)
        fixed = max(drift_scores(frozen_edges, frozen, now)[s].psi for s in CONFIDENCE_SIGNALS)
        rolling = max(drift_scores(before_edges, before, now)[s].psi for s in CONFIDENCE_SIGNALS)
        print(f"{day:11d}{fixed:17.4f}{rolling:18.4f}")
    print("The rolling baseline sees the new template arrive, then learns it: a month later the")
    print("change is the baseline, and a second change on top of it would be measured from there.")


_try("rolling baseline", _show_rolling_baseline, needs=("exercise 1", "exercise 2", "exercise 3"))

## 13. Self-check

1. The template change pushed a confidence signal's PSI past `PSI_RULE_OF_THUMB`, and the
   audited defect rate did not move. The monitor should:
   - (a) page: PSI above the rule-of-thumb band means action is required
   - (b) retrain the extractor on the new template before it degrades
   - (c) record a template change and not page: the extractor is less confident, not less
         correct

2. Why did the confidence-proxy pager never see the regression?
   - (a) the regression's wrong values were written with the confidence of correct ones, so
         the confidence distribution did not move
   - (b) the confidence proxy averages over too few documents a day
   - (c) its threshold was chosen against a looser budget than the audit's

3. A handbook's rule of thumb gives `h` around 4 or 5. The reason this lesson measures instead
   of adopting it is:
   - (a) round numbers are always too low for a document pipeline
   - (b) a threshold's false-page rate depends on `k` and on how the proxy behaves day to day,
         so only a measurement on no-change data can say what it costs against the budget
   - (c) the handbook assumes a different CUSUM recursion

4. A colleague proposes scoring each month against the month before it, edges re-cut each
   time, "so the baseline stays current". What happens to the template change?
   - (a) nothing changes, because PSI is symmetric in its two samples
   - (b) only the categorical signals are affected
   - (c) it shows up for a month or so, then the new template is the baseline and the change
         disappears from the report

5. A page arrives in a week whose drift report also shows confidence moved. The triage is:
   - (a) page: quality moved, and a change that breaks extraction is a regression whatever
         else moved with it
   - (b) do not page: the confidence drift explains the quality change
   - (c) wait for next week's report before deciding

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

A drift report that says which distribution moved, on edges that cannot follow the drift; a
quality proxy that can see a confidently wrong model; a CUSUM that pages on it; and a threshold
that states what it costs. The programme's capstone is specified to assemble this card into its
conformity pack. The audit that feeds the pager is people reading documents, and module 9's
router, which has a human tier, is where that kind of work gets a price.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_psi),
                              ("exercise 2", _check_kl),
                              ("exercise 3", _check_edges_and_drift),
                              ("exercise 4", _check_defect_rate),
                              ("exercise 5", _check_cusum),
                              ("exercise 6", _check_threshold),
                              ("exercise 7", _check_triage)):
            _try(_name, _check)
    _progress_board()
    _wall = time.perf_counter() - _LESSON_T0
    # Whole seconds: two machines disagree at the first decimal, and that is noise, not a result.
    print("\nlesson wall time so far: " + ("under a second" if _wall < 1 else f"{_wall:.0f}s"))
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.